<a href="https://colab.research.google.com/github/gorniakgrzegorz/publikacje/blob/main/Torun_GeoAI_2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analiza zmian pokrycia terenu w północnym Toruniu (2015–2025)

Klasyfikacja pokrycia terenu metodą Dynamic World na podstawie zobrazowań Sentinel-2,
analiza zmian metodą porównania poklasyfikacyjnego oraz walidacja względem Urban Atlas 2021.

**Obszar:** dzielnice Wrzosy, Bielawy i JAR (północny Toruń), około 28,2 km².
**Środowisko:** Google Earth Engine + Python.

Notatnik należy uruchamiać komórka po komórce. Autoryzacja Google Earth Engine wymagana
jest w komórce 2; plik wektorowy Urban Atlas (.fgb) wczytywany jest w komórce 12.

## 1. Instalacja bibliotek

Instalacja pakietów do obsługi Google Earth Engine, map interaktywnych oraz danych wektorowych.

In [ ]:
# Instalacja bibliotek
!pip install earthengine-api geemap geopandas pyogrio --quiet
print("Biblioteki zainstalowane.")

## 2. Autoryzacja Google Earth Engine

Pierwsze uruchomienie otwiera okno logowania do konta Google z dostępem do Earth Engine.

In [ ]:
import ee
import geemap

try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()

print("Google Earth Engine gotowy.")

## 3. Definicja obszaru badań

Granice obszaru badań zdefiniowano jako wielokąt w układzie współrzędnych geograficznych
WGS84 (EPSG:4326), o współrzędnych granicznych od 53,0233°N, 18,5558°E do
53,0645°N, 18,6459°E. Powierzchnia obszaru wynosi około 28,2 km².

In [ ]:
# Obszar badań: północne dzielnice Torunia (Wrzosy, Bielawy, JAR)
TORUN_POLNOC = ee.Geometry.Polygon([[
    [18.5558, 53.0233],
    [18.6459, 53.0233],
    [18.6459, 53.0645],
    [18.5558, 53.0645],
    [18.5558, 53.0233]
]])

# Obszar pomocniczy (cały Toruń) do wizualizacji szerszego kontekstu
TORUN_CALY = ee.Geometry.Rectangle([18.47, 52.94, 18.76, 53.10])

pow_km2 = TORUN_POLNOC.area().divide(1e6).getInfo()
print(f"Powierzchnia obszaru badań: {pow_km2:.2f} km2")

## 4. Maskowanie chmur metodą Cloud Score+

Do maskowania chmur i cieni chmur zastosowano Cloud Score+, produkt oceny jakości pikseli
Sentinel-2 dostępny w katalogu Google Earth Engine. Cloud Score+ dostarcza ciągły wskaźnik
przydatności piksela (pasmo `cs_cdf`) w zakresie od 0 do 1; przyjęto próg 0,60, mieszczący
się w zalecanym przedziale 0,50–0,65. Pasmo QA60, stosowane w starszych procedurach, nie jest
wiarygodne dla scen pozyskanych po styczniu 2022 roku, dlatego nie zostało użyte.

In [ ]:
# Kolekcja Cloud Score+ (ocena jakości pikseli Sentinel-2)
CS_PLUS = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
CS_BAND = 'cs_cdf'
PROG_CZYSTOSCI = 0.60

def maskuj_chmury_csplus(img):
    """Maskuje chmury i cienie metodą Cloud Score+ (cs_cdf >= 0.60)."""
    return img.updateMask(img.select(CS_BAND).gte(PROG_CZYSTOSCI)).divide(10000)

def kompozyt_s2(geom, rok, miesiac_start, miesiac_koniec, prog_chmur=60):
    """Tworzy medianowy kompozyt Sentinel-2 z maskowaniem Cloud Score+."""
    data_start = f'{rok}-{miesiac_start:02d}-01'
    data_koniec = f'{rok}-{miesiac_koniec:02d}-28'

    kolekcja = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(geom)
        .filterDate(data_start, data_koniec)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', prog_chmur))
        .linkCollection(CS_PLUS, [CS_BAND])
        .map(maskuj_chmury_csplus)
    )

    n = kolekcja.size().getInfo()
    print(f"  {rok} (m-ce {miesiac_start}-{miesiac_koniec}): {n} zobrazowań")

    if n == 0:
        print("  Brak zobrazowań - poszerzam zakres czasowy...")
        kolekcja = (
            ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(geom).filterDate(f'{rok}-06-01', f'{rok}-10-28')
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80))
            .linkCollection(CS_PLUS, [CS_BAND]).map(maskuj_chmury_csplus)
        )
        n = kolekcja.size().getInfo()
        print(f"  Po poszerzeniu: {n} zobrazowań")

    return kolekcja.median().clip(geom)

print("Funkcje Sentinel-2 (Cloud Score+) gotowe.")

## 5. Kompozyty Sentinel-2

Dla wszystkich analizowanych lat przyjęto jednolity okres kompozytowania obejmujący sezon
wegetacyjny (czerwiec–wrzesień), co zapewnia porównywalność warunków fenologicznych.

In [ ]:
# Jednolity sezon wegetacyjny dla wszystkich lat
MIES_START, MIES_KONIEC = 6, 9

print("Tworzenie kompozytów Sentinel-2:")
s2_2015 = kompozyt_s2(TORUN_POLNOC, 2015, MIES_START, MIES_KONIEC)
s2_2021 = kompozyt_s2(TORUN_POLNOC, 2021, MIES_START, MIES_KONIEC)
s2_2025 = kompozyt_s2(TORUN_POLNOC, 2025, MIES_START, MIES_KONIEC)
print("Kompozyty gotowe.")

Podgląd kompozytów na mapie interaktywnej (kompozycja barw naturalnych RGB).

In [ ]:
VIS_RGB = {'min': 0.0, 'max': 0.3, 'bands': ['B4', 'B3', 'B2']}

Mapa = geemap.Map()
Mapa.centerObject(TORUN_POLNOC, 13)
Mapa.addLayer(s2_2015, VIS_RGB, 'Sentinel-2 2015')
Mapa.addLayer(s2_2025, VIS_RGB, 'Sentinel-2 2025')
Mapa.addLayer(TORUN_POLNOC, {'color': 'red'}, 'Granica obszaru badań')
Mapa

## 6. Klasyfikacja pokrycia terenu metodą Dynamic World

Dynamic World dostarcza dla każdego piksela rozkład prawdopodobieństwa przynależności do
dziewięciu klas pokrycia terenu oraz etykietę klasy o najwyższym prawdopodobieństwie.
Dla każdego roku wyznaczono etykietę dominującą (najczęstszą w sezonie) oraz średnie
prawdopodobieństwo klasy zwycięskiej jako miarę pewności klasyfikacji. Piksele o pewności
poniżej 0,50 oznaczono jako obszary podwyższonej niepewności.

In [ ]:
NAZWY_KLAS = ['woda', 'drzewa', 'trawa', 'roslinnosc_podmokla',
              'uprawy', 'krzewy', 'zabudowa', 'gleba_odkryta', 'snieg_lod']
NAZWY_KLAS_PL = ['Woda', 'Las/drzewa', 'Trawa i laki', 'Rosl. podmokla',
                 'Uprawy rolne', 'Krzewy', 'Zabudowa', 'Gleba odkryta', 'Snieg']
PASMA_PRAWD = ['water', 'trees', 'grass', 'flooded_vegetation', 'crops',
               'shrub_and_scrub', 'built', 'bare', 'snow_and_ice']

PROG_PEWNOSCI = 0.50

def klasyfikacja_dw(geom, rok, miesiac_start=6, miesiac_koniec=9):
    """Zwraca etykietę klasy dominującej, mapę pewności i maskę obszarów niepewnych."""
    data_start = f'{rok}-{miesiac_start:02d}-01'
    data_koniec = f'{rok}-{miesiac_koniec:02d}-28'
    kol = (ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')
           .filterBounds(geom).filterDate(data_start, data_koniec))
    n = kol.size().getInfo()
    print(f"  {rok}: {n} klasyfikacji Dynamic World")

    etykieta = kol.select('label').mode().clip(geom)
    prawd_srednie = kol.select(PASMA_PRAWD).mean().clip(geom)
    pewnosc = prawd_srednie.reduce(ee.Reducer.max()).rename('pewnosc')
    maska_niepewna = pewnosc.lt(PROG_PEWNOSCI).rename('niepewnosc')

    return {'etykieta': etykieta, 'pewnosc': pewnosc, 'niepewna': maska_niepewna}

print("Klasyfikacja Dynamic World:")
dw15 = klasyfikacja_dw(TORUN_POLNOC, 2015)
dw21 = klasyfikacja_dw(TORUN_POLNOC, 2021)
dw25 = klasyfikacja_dw(TORUN_POLNOC, 2025)

dw_2015 = dw15['etykieta']
dw_2021 = dw21['etykieta']
dw_2025 = dw25['etykieta']
print("Gotowe: etykiety i mapy pewności dla 2015, 2021, 2025.")

Statystyki pewności klasyfikacji: średnia pewność oraz udział pikseli o obniżonej pewności.

In [ ]:
def statystyki_pewnosci(dw_dict, rok):
    sr = dw_dict['pewnosc'].reduceRegion(
        ee.Reducer.mean(), TORUN_POLNOC, 10, maxPixels=1e10).get('pewnosc').getInfo()
    udzial = dw_dict['niepewna'].reduceRegion(
        ee.Reducer.mean(), TORUN_POLNOC, 10, maxPixels=1e10).get('niepewnosc').getInfo()
    print(f"  {rok}: srednia pewnosc = {sr:.3f}, udzial pikseli niepewnych = {udzial*100:.1f}%")
    return sr, udzial

print("Pewność klasyfikacji Dynamic World:")
statystyki_pewnosci(dw15, 2015)
statystyki_pewnosci(dw21, 2021)
statystyki_pewnosci(dw25, 2025)

## 7. Powierzchnie klas i analiza zmian

Powierzchnię każdej klasy obliczono jako sumę pól pikseli należących do tej klasy.
Klasę śniegu i lodu wykluczono z analizy, ponieważ kompozyty obejmują sezon wegetacyjny.

In [ ]:
import pandas as pd

KLASY_ANALIZA = list(range(8))   # klasy 0-7, śnieg (8) wykluczony

def powierzchnie_klas(etykieta, rok):
    """Liczy powierzchnię [ha] każdej klasy 0-7."""
    pole = ee.Image.pixelArea()
    wynik = {}
    for k in KLASY_ANALIZA:
        ha = (etykieta.eq(k).multiply(pole)
              .reduceRegion(ee.Reducer.sum(), TORUN_POLNOC, 10, maxPixels=1e10)
              .get('label'))
        wynik[NAZWY_KLAS_PL[k]] = ee.Number(ha).divide(1e4).getInfo()
    return wynik

print("Liczenie powierzchni klas...")
p2015 = powierzchnie_klas(dw_2015, 2015)
p2025 = powierzchnie_klas(dw_2025, 2025)

df = pd.DataFrame({
    'Klasa': list(p2015.keys()),
    '2015 [ha]': [round(p2015[k], 2) for k in p2015],
    '2025 [ha]': [round(p2025[k], 2) for k in p2015],
})
df['Zmiana [ha]'] = (df['2025 [ha]'] - df['2015 [ha]']).round(2)
df['Zmiana [%]'] = ((df['Zmiana [ha]'] / df['2015 [ha]'].replace(0, float('nan'))) * 100).round(1)
df.to_csv('torun_tabela_zmian.csv', index=False)
print(df.to_string(index=False))

Syntetyczne wskaźniki środowiskowe: powierzchnia zabudowy, wskaźnik uszczelnienia oraz
bilans terenów biologicznie czynnych.

In [ ]:
suma_2015 = df['2015 [ha]'].sum()
suma_2025 = df['2025 [ha]'].sum()
zab_2015 = df.loc[df['Klasa']=='Zabudowa', '2015 [ha]'].values[0]
zab_2025 = df.loc[df['Klasa']=='Zabudowa', '2025 [ha]'].values[0]

bio_klasy = ['Las/drzewa', 'Trawa i laki', 'Rosl. podmokla', 'Uprawy rolne', 'Krzewy']
bio_2015 = df[df['Klasa'].isin(bio_klasy)]['2015 [ha]'].sum()
bio_2025 = df[df['Klasa'].isin(bio_klasy)]['2025 [ha]'].sum()

print(f"AOI (suma klas): {suma_2025:.1f} ha")
print(f"Zabudowa: {zab_2015:.1f} -> {zab_2025:.1f} ha (zmiana {zab_2025-zab_2015:+.1f} ha)")
print(f"Uszczelnienie: {zab_2015/suma_2015*100:.1f}% -> {zab_2025/suma_2025*100:.1f}%")
print(f"Tereny biol. czynne: {bio_2015:.1f} -> {bio_2025:.1f} ha (zmiana {bio_2025-bio_2015:+.1f} ha)")

Pochodzenie nowej zabudowy: identyfikacja klas pokrycia z 2015 roku, które do 2025 roku
zostały zabudowane.

In [ ]:
KOD_ZAB = 6
nowa_zabudowa = dw_2015.neq(KOD_ZAB).And(dw_2025.eq(KOD_ZAB))

pole = ee.Image.pixelArea()
wynik_pochodzenie = {}
for k in KLASY_ANALIZA:
    if k == KOD_ZAB:
        continue
    maska = nowa_zabudowa.And(dw_2015.eq(k))
    ha = (maska.multiply(pole).reduceRegion(
        ee.Reducer.sum(), TORUN_POLNOC, 10, maxPixels=1e10).get('label'))
    wynik_pochodzenie[NAZWY_KLAS_PL[k]] = round(ee.Number(ha).divide(1e4).getInfo(), 2)

df_poch = pd.DataFrame({
    'Klasa zrodlowa (2015)': list(wynik_pochodzenie.keys()),
    'Powierzchnia [ha]': list(wynik_pochodzenie.values()),
})
suma_nowa = df_poch['Powierzchnia [ha]'].sum()
df_poch['Udzial [%]'] = (df_poch['Powierzchnia [ha]'] / suma_nowa * 100).round(1)
df_poch = df_poch.sort_values('Powierzchnia [ha]', ascending=False)
df_poch.to_csv('torun_pochodzenie_zabudowy.csv', index=False)
print(df_poch.to_string(index=False))
print(f"\nLaczna nowa zabudowa: {suma_nowa:.2f} ha")

## 8. Generowanie rycin

W tej części powstają trzy ryciny ilustrujące wyniki analizy zmian: porównanie powierzchni
klas, zmiana netto powierzchni oraz bilans terenów biologicznie czynnych względem zabudowy.
Każda rycina zapisywana jest w rozdzielczości 300 dpi w formacie PNG.

### 8.1. Konfiguracja stylu wykresów

Ustawienie czcionki szeryfowej i jednolitego rozmiaru tekstu, spójnego z dokumentem tekstowym.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.family'] = 'DejaVu Serif'
plt.rcParams['font.size'] = 10

# Klasy o znaczącej powierzchni (pomijamy klasy zerowe: woda, podmokła, śnieg)
klasy_wykres = ['Las/drzewa', 'Zabudowa', 'Krzewy', 'Uprawy rolne', 'Trawa i laki', 'Gleba odkryta']
etykiety = ['Lasy i drzewa', 'Zabudowa', 'Krzewy', 'Uprawy rolne', 'Trawy i łąki', 'Gleba odkryta']

# Wartości z tabeli zmian
v2015 = [df.loc[df['Klasa']==k, '2015 [ha]'].values[0] for k in klasy_wykres]
v2025 = [df.loc[df['Klasa']==k, '2025 [ha]'].values[0] for k in klasy_wykres]
zmiana = [df.loc[df['Klasa']==k, 'Zmiana [ha]'].values[0] for k in klasy_wykres]
print("Dane do rycin przygotowane.")

### 8.2. Rycina 1 — porównanie powierzchni klas w latach 2015 i 2025

Wykres słupkowy zestawiający powierzchnię każdej klasy w obu latach granicznych.

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.2))
x = np.arange(len(klasy_wykres)); w = 0.38
ax.bar(x - w/2, v2015, w, label='2015', color='#5B7C99', edgecolor='black', linewidth=0.4)
ax.bar(x + w/2, v2025, w, label='2025', color='#C4281B', edgecolor='black', linewidth=0.4)
ax.set_xticks(x); ax.set_xticklabels(etykiety, rotation=25, ha='right')
ax.set_ylabel('Powierzchnia [ha]')
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('rycina1_porownanie_klas.png', dpi=300, bbox_inches='tight')
plt.show()
print("Zapisano: rycina1_porownanie_klas.png")

### 8.3. Rycina 2 — zmiana netto powierzchni klas

Wykres poziomy przedstawiający kierunek i wielkość zmiany każdej klasy. Kolor czerwony
oznacza przyrost, zielony ubytek.

In [ ]:
order = np.argsort(zmiana)
zm = [zmiana[i] for i in order]
nz = [etykiety[i] for i in order]
kol = ['#C4281B' if z > 0 else '#397D49' for z in zm]

fig, ax = plt.subplots(figsize=(7.2, 4.0))
ax.barh(range(len(zm)), zm, color=kol, edgecolor='black', linewidth=0.4)
ax.set_yticks(range(len(zm))); ax.set_yticklabels(nz)
ax.axvline(0, color='black', linewidth=0.6)
ax.set_xlabel('Zmiana powierzchni 2015–2025 [ha]')
margines = max(abs(min(zm)), abs(max(zm))) * 0.18
ax.set_xlim(min(zm) - margines*2, max(zm) + margines*2)
for i, z in enumerate(zm):
    ax.text(z + (margines*0.3 if z > 0 else -margines*0.3), i, f'{z:+.1f}',
            va='center', ha='left' if z > 0 else 'right', fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('rycina2_zmiana_netto.png', dpi=300, bbox_inches='tight')
plt.show()
print("Zapisano: rycina2_zmiana_netto.png")

### 8.4. Rycina 3 — bilans terenów biologicznie czynnych i zabudowy

Zestawienie sumarycznej powierzchni terenów biologicznie czynnych oraz zabudowy w obu latach.

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 4.2))
kat = ['Tereny\nbiologicznie\nczynne', 'Zabudowa']
val15 = [bio_2015, zab_2015]; val25 = [bio_2025, zab_2025]
x = np.arange(2); w = 0.38
b1 = ax.bar(x - w/2, val15, w, label='2015', color='#88B053', edgecolor='black', linewidth=0.4)
b2 = ax.bar(x + w/2, val25, w, label='2025', color='#A0522D', edgecolor='black', linewidth=0.4)
ax.set_xticks(x); ax.set_xticklabels(kat)
ax.set_ylabel('Powierzchnia [ha]')
ax.legend(frameon=False)
for b in list(b1) + list(b2):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 8, f'{b.get_height():.0f}',
            ha='center', fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('rycina3_bilans.png', dpi=300, bbox_inches='tight')
plt.show()
print("Zapisano: rycina3_bilans.png")

## 9. Walidacja względem Urban Atlas 2021

Walidacja odnosi się do roku 2021, zgodnego czasowo z produktem Urban Atlas, i stanowi ocenę
wiarygodności procedury klasyfikacyjnej. Urban Atlas 2021 to wektorowy produkt Copernicus
o nomenklaturze 27 klas (17 klas miejskich z minimalną jednostką kartowania 0,25 ha oraz
klasy wiejskie z jednostką 1 ha), w układzie EPSG:3035.

### 9.1. Wczytanie warstwy Urban Atlas

Po uruchomieniu komórki należy wskazać plik wektorowy Urban Atlas w formacie FlatGeobuf (.fgb).

In [ ]:
from google.colab import files
print("Wskaż plik Urban Atlas (.fgb)...")
uploaded = files.upload()
import os
nazwa_ua = list(uploaded.keys())[0]

import geopandas as gpd
ua = gpd.read_file(nazwa_ua)
print(f"Poligonów: {len(ua):,} | CRS: {ua.crs}")
print(ua['class_2021'].value_counts().to_string())

### 9.2. Przycięcie do obszaru badań i harmonizacja legend

Warstwę przycięto do obszaru badań, a następnie obie legendy (Dynamic World i Urban Atlas)
sprowadzono do wspólnego schematu sześcioklasowego za pomocą tabeli przejścia.

In [ ]:
from shapely.geometry import box

AOI_WSP = [18.5558, 53.0233, 18.6459, 53.0645]
aoi_3035 = gpd.GeoDataFrame({'geometry': [box(*AOI_WSP)]}, crs='EPSG:4326').to_crs(ua.crs)
ua_aoi = gpd.clip(ua, aoi_3035)
ua_aoi = ua_aoi[~ua_aoi.geometry.is_empty & ua_aoi.geometry.notna()].copy()
ua_aoi['pow_ha'] = ua_aoi.geometry.area / 1e4
print(f"Poligonów po przycięciu: {len(ua_aoi):,}")

# Wspólna legenda 6-klasowa
LEGENDA_WSP = {1: 'Zabudowa', 2: 'Lasy i drzewa', 3: 'Tereny trawiaste',
               4: 'Uprawy rolne', 5: 'Wody i podmokłe', 6: 'Gleba odkryta'}

# Tabela przejścia: Dynamic World (0-7) -> wspólna legenda
DW_NA_WSP = {0: 5, 1: 2, 2: 3, 3: 5, 4: 4, 5: 3, 6: 1, 7: 6}

def ua_na_wsp(nazwa):
    """Mapuje nazwę klasy Urban Atlas na kod wspólnej legendy.
    Klasy trawiaste sprawdzane przed klasami zabudowy."""
    n = str(nazwa).lower()
    if any(s in n for s in ['green urban', 'sports', 'leisure', 'pasture', 'herbaceous', 'grassland']):
        return 3
    if any(s in n for s in ['urban fabric', 'industrial', 'road', 'railway', 'port areas',
                            'airport', 'construction', 'mineral', 'dump', 'isolated structures',
                            'without current use']):
        return 1
    if 'forest' in n: return 2
    if any(s in n for s in ['arable', 'permanent crop', 'complex', 'cultivation']): return 4
    if any(s in n for s in ['water', 'wetland']): return 5
    if any(s in n for s in ['open spaces', 'little or no vegetation', 'bare']): return 6
    return 0

ua_aoi['kod_wsp'] = ua_aoi['class_2021'].apply(ua_na_wsp)
ua_walid = ua_aoi[ua_aoi['kod_wsp'] > 0].copy()
print("Mapowanie klas Urban Atlas:")
for kl in sorted(ua_aoi['class_2021'].unique()):
    print(f"  [{ua_na_wsp(kl)}] {LEGENDA_WSP.get(ua_na_wsp(kl), 'pominięte')} <- {kl}")

### 9.3. Przesłanie do Earth Engine i rasteryzacja

Poligony Urban Atlas przesłano do Earth Engine i zrasteryzowano do siatki 10 m, zgodnej
z rozdzielczością Dynamic World. Wyznaczono również strefę przygraniczną poligonów, która
w ocenie dokładności analizowana jest oddzielnie.

In [ ]:
import json as _json
ua_gee = ua_walid.to_crs('EPSG:4326')[['kod_wsp', 'geometry']].copy()
ua_gee['geometry'] = ua_gee.geometry.simplify(0.00002, preserve_topology=True)

def gdf_do_ee(gdf, porcja=400):
    czesci = []
    for s in range(0, len(gdf), porcja):
        feats = []
        for _, w in gdf.iloc[s:s+porcja].iterrows():
            gj = _json.loads(gpd.GeoSeries([w.geometry]).to_json())['features'][0]['geometry']
            feats.append(ee.Feature(ee.Geometry(gj), {'kod_wsp': int(w['kod_wsp'])}))
        czesci.append(ee.FeatureCollection(feats))
        print(f"  {min(s+porcja, len(gdf))}/{len(gdf)} poligonów")
    fc = czesci[0]
    for c in czesci[1:]:
        fc = fc.merge(c)
    return fc

print("Przesyłanie Urban Atlas do Earth Engine...")
ua_fc = gdf_do_ee(ua_gee)

ua_raster = ee.Image(0).paint(ua_fc, 'kod_wsp').rename('ua_klasa').clip(TORUN_POLNOC)
ua_raster = ua_raster.updateMask(ua_raster.gt(0))

krawedzie = ua_raster.unmask(0).focal_max(1).neq(ua_raster.unmask(0).focal_min(1))
maska_brzegu = krawedzie.focal_max(1)
print("Rasteryzacja zakończona.")

### 9.4. Próba walidacyjna

Przekodowano klasyfikację Dynamic World 2021 do wspólnej legendy, a następnie pobrano
stratyfikowaną próbę losową obejmującą 500 punktów na klasę (około 3000 punktów łącznie),
ze stałym ziarnem generatora liczb losowych zapewniającym powtarzalność.

In [ ]:
stare = list(DW_NA_WSP.keys()); nowe = [DW_NA_WSP[k] for k in stare]
dw_2021_wsp = dw_2021.remap(stare, nowe).rename('dw_klasa').clip(TORUN_POLNOC)

porownanie = (dw_2021_wsp
              .addBands(ua_raster.rename('ua_klasa'))
              .addBands(maska_brzegu.rename('brzeg')))

PKT_NA_KLASE = 500
proba = ua_raster.rename('strata').addBands(porownanie).stratifiedSample(
    numPoints=PKT_NA_KLASE, classBand='strata', region=TORUN_POLNOC,
    scale=10, seed=42, geometries=False)
print(f"Pobrano {proba.size().getInfo():,} punktów (seed=42).")

dane = proba.select(['dw_klasa', 'ua_klasa', 'brzeg']).getInfo()['features']
dfw = pd.DataFrame([f['properties'] for f in dane]).dropna().astype(int)
dfw = dfw[(dfw['dw_klasa'] > 0) & (dfw['ua_klasa'] > 0)]
dfw_core = dfw[dfw['brzeg'] == 0]
print(f"Punktów łącznie: {len(dfw):,} | rdzeniowych (poza strefą granic): {len(dfw_core):,}")

### 9.5. Macierz pomyłek i miary dokładności

Dla próby pełnej oraz rdzeniowej wyznaczono dokładność całkowitą, współczynnik kappa Cohena
oraz dokładność producenta, dokładność użytkownika i miarę F1 dla każdej klasy.

In [ ]:
def metryki(dframe, opis):
    klasy = sorted(set(dframe['ua_klasa']) | set(dframe['dw_klasa']))
    nazwy = [LEGENDA_WSP[k] for k in klasy]
    idx = {k: i for i, k in enumerate(klasy)}
    n = len(klasy)
    M = np.zeros((n, n), dtype=int)
    for _, w in dframe.iterrows():
        M[idx[w['ua_klasa']], idx[w['dw_klasa']]] += 1

    suma = M.sum(); diag = np.diag(M)
    OA = diag.sum() / suma
    sw = M.sum(1); sk = M.sum(0)
    pe = (sw*sk).sum() / suma**2
    kappa = (OA - pe) / (1 - pe)
    PA = np.divide(diag, sw, out=np.zeros(n), where=sw>0)
    UA = np.divide(diag, sk, out=np.zeros(n), where=sk>0)
    F1 = np.divide(2*PA*UA, PA+UA, out=np.zeros(n), where=(PA+UA)>0)

    print(f"\n=== {opis} ===")
    print(f"Punktów: {suma} | OA = {OA*100:.1f}% | kappa = {kappa:.3f}")
    tab = pd.DataFrame({'Klasa': nazwy, 'N': sw,
        'PA [%]': (PA*100).round(1), 'UA [%]': (UA*100).round(1), 'F1 [%]': (F1*100).round(1)})
    print(tab.to_string(index=False))
    return OA, kappa, tab, pd.DataFrame(M, index=nazwy, columns=nazwy)

OA_all, kappa_all, tab_all, mat_all = metryki(dfw, "Wszystkie punkty")
OA_core, kappa_core, tab_core, mat_core = metryki(dfw_core, "Punkty rdzeniowe")

mat_all.to_csv('walidacja_macierz.csv')
tab_all.to_csv('walidacja_metryki_klasy.csv', index=False)
pd.DataFrame({'Metryka': ['OA_all [%]', 'kappa_all', 'OA_core [%]', 'kappa_core', 'N_all', 'N_core'],
              'Wartosc': [round(OA_all*100,1), round(kappa_all,3), round(OA_core*100,1),
                          round(kappa_core,3), len(dfw), len(dfw_core)]}
            ).to_csv('walidacja_metryki_globalne.csv', index=False)
print("\nZapisano pliki CSV walidacji.")

### 9.6. Rycina 4 — macierz pomyłek

Heatmapa macierzy pomyłek znormalizowanej wierszami (procent referencji Urban Atlas).

In [ ]:
M = mat_all.values
proc = M / M.sum(1, keepdims=True) * 100
nazwy = list(mat_all.index)

fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(proc, cmap='YlGnBu', vmin=0, vmax=100)
ax.set_xticks(range(len(nazwy))); ax.set_yticks(range(len(nazwy)))
ax.set_xticklabels(nazwy, rotation=45, ha='right'); ax.set_yticklabels(nazwy)
ax.set_xlabel('Dynamic World'); ax.set_ylabel('Urban Atlas (referencja)')
ax.set_title(f'Macierz pomyłek DW 2021 vs Urban Atlas 2021\nOA={OA_all*100:.1f}%, kappa={kappa_all:.3f}')
for i in range(len(nazwy)):
    for j in range(len(nazwy)):
        ax.text(j, i, f'{proc[i,j]:.0f}', ha='center', va='center',
                color='white' if proc[i,j] > 50 else 'black', fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.8, label='% referencji')
plt.tight_layout()
plt.savefig('rycina4_macierz_pomylek.png', dpi=300, bbox_inches='tight')
plt.show()
print("Zapisano: rycina4_macierz_pomylek.png")

## 10. Eksport wyników

Eksport map rastrowych do Google Drive oraz pobranie tabel CSV i rycin PNG na dysk lokalny.

In [ ]:
for obraz, nazwa in [(dw_2015.toByte(), 'dw_2015'), (dw_2025.toByte(), 'dw_2025'),
                     (nowa_zabudowa.toByte(), 'nowa_zabudowa'),
                     (dw25['pewnosc'].multiply(100).toByte(), 'pewnosc_2025'),
                     (ua_raster.toByte(), 'urban_atlas_2021_raster')]:
    ee.batch.Export.image.toDrive(
        image=obraz, description=f'torun_{nazwa}', folder='torun_geoai',
        scale=10, region=TORUN_POLNOC, crs='EPSG:2180', maxPixels=1e10).start()
print("Eksport map uruchomiony (zakładka Tasks).")

In [ ]:
from google.colab import files
for f in ['torun_tabela_zmian.csv', 'torun_pochodzenie_zabudowy.csv',
          'walidacja_macierz.csv', 'walidacja_metryki_klasy.csv',
          'walidacja_metryki_globalne.csv',
          'rycina1_porownanie_klas.png', 'rycina2_zmiana_netto.png',
          'rycina3_bilans.png', 'rycina4_macierz_pomylek.png']:
    files.download(f)